<a href="https://colab.research.google.com/github/Reshsajee/my-project-resh/blob/main/mobilenetv2%2Bsvm_hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

In [ ]:
import os
import numpy as np
import cv2

X = []
y = []

dataset_path = "/content/drive/MyDrive/R11/MG"

for folder in sorted(os.listdir(dataset_path)):
    folder_path = os.path.join(dataset_path, folder)

    if not os.path.isdir(folder_path):
        continue

    images = sorted(os.listdir(folder_path))

    for i, img_name in enumerate(images):
        img_path = os.path.join(folder_path, img_name)

        img = cv2.imread(img_path)
        img = cv2.resize(img, (224, 224))

        X.append(img)
        y.append(i)  # 0=pinhead,1=growing,2=mature

X = np.array(X)
y = np.array(y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
X_train = tf.keras.applications.mobilenet_v2.preprocess_input(X_train)
X_test = tf.keras.applications.mobilenet_v2.preprocess_input(X_test)

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

# Custom head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(3, activation='softmax')(x)

model_mobilenet = Model(inputs=base_model.input, outputs=output)

model_mobilenet.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_mobilenet.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
history = model_mobilenet.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=15,
    batch_size=8
)

Epoch 1/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 13s 655ms/step - accuracy: 0.3816 - loss: 1.4287 - val_accuracy: 0.7000 - val_loss: 0.8910
Epoch 2/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 436ms/step - accuracy: 0.7237 - loss: 0.6455 - val_accuracy: 0.7000 - val_loss: 0.7236
Epoch 3/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 387ms/step - accuracy: 0.8158 - loss: 0.4024 - val_accuracy: 0.6500 - val_loss: 1.0039
Epoch 4/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 588ms/step - accuracy: 0.8947 - loss: 0.2862 - val_accuracy: 0.6500 - val_loss: 0.6878
Epoch 5/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 392ms/step - accuracy: 0.9342 - loss: 0.1712 - val_accuracy: 0.6500 - val_loss: 1.1368
Epoch 6/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 381ms/step - accuracy: 0.9079 - loss: 0.2113 - val_accuracy: 0.6500 - val_loss: 0.7989
Epoch 7/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 651ms/step - accuracy: 0.9605 - loss: 0.1261 - val_accuracy: 0.6500 - val_loss: 1.1262
Epoch 8/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 379ms/step - accuracy: 0.9474 - loss: 0.1785 - val_accuracy: 0

In [ ]:
loss, accuracy = model_mobilenet.evaluate(X_test, y_test)

print("✅ Test Accuracy:", accuracy * 100, "%")
print("✅ Test Loss:", loss)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.7000 - loss: 1.2349
✅ Test Accuracy: 69.9999988079071 %
✅ Test Loss: 1.2349045276641846


In [ ]:
for layer in model_mobilenet.layers[-30:]:
    layer.trainable = True

In [ ]:
model_mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),  # low LR
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_fine = model_mobilenet.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=8
)

Epoch 1/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 21s 929ms/step - accuracy: 0.5395 - loss: 1.9089 - val_accuracy: 0.6500 - val_loss: 1.1991
Epoch 2/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 547ms/step - accuracy: 0.5658 - loss: 1.6305 - val_accuracy: 0.6000 - val_loss: 1.1830
Epoch 3/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 596ms/step - accuracy: 0.6711 - loss: 1.1974 - val_accuracy: 0.6000 - val_loss: 1.1561
Epoch 4/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 10s 549ms/step - accuracy: 0.6447 - loss: 1.0614 - val_accuracy: 0.6000 - val_loss: 1.1532
Epoch 5/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 723ms/step - accuracy: 0.7500 - loss: 0.8117 - val_accuracy: 0.6000 - val_loss: 1.1456
Epoch 6/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 491ms/step - accuracy: 0.7632 - loss: 0.7452 - val_accuracy: 0.6000 - val_loss: 1.1542
Epoch 7/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 492ms/step - accuracy: 0.8158 - loss: 0.6137 - val_accuracy: 0.6000 - val_loss: 1.1721
Epoch 8/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 736ms/step - accuracy: 0.8158 - loss: 0.4399 - val_accuracy: 

In [ ]:
import os
from datetime import datetime
import cv2
import numpy as np

X = []
y_time = []   # time to harvest (in hours)

dataset_path = "/content/drive/MyDrive/R11/MG"

for folder in sorted(os.listdir(dataset_path)):
    folder_path = os.path.join(dataset_path, folder)

    if not os.path.isdir(folder_path):
        continue

    images = sorted(os.listdir(folder_path))

    times = []

    # Extract timestamps
    for img_name in images:
        time_str = img_name.split('_')[1] + "_" + img_name.split('_')[2].split('.')[0]
        time_obj = datetime.strptime(time_str, "%Y%m%d_%H%M%S")
        times.append(time_obj)

    harvest_time = times[-1]  # last image = mature

    # Loop again for training
    for i, img_name in enumerate(images):
        img_path = os.path.join(folder_path, img_name)

        img = cv2.imread(img_path)
        img = cv2.resize(img, (224, 224))

        X.append(img)

        # ⏳ remaining time in HOURS
        remaining_time = (harvest_time - times[i]).total_seconds() / 3600.0
        y_time.append(remaining_time)

X = np.array(X)
y_time = np.array(y_time)

print("Loaded:", X.shape, y_time.shape)

Loaded: (96, 224, 224, 3) (96,)


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import tensorflow as tf

# Preprocess
X = tf.keras.applications.mobilenet_v2.preprocess_input(X)

# Base model
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)

# 🔥 regression output
output = Dense(1, activation='linear')(x)

model_time = Model(inputs=base_model.input, outputs=output)

model_time.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

model_time.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_time, test_size=0.2, random_state=42
)

history = model_time.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=8
)

Epoch 1/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 14s 690ms/step - loss: 3955.2783 - mae: 49.3952 - val_loss: 4401.2827 - val_mae: 54.1227
Epoch 2/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 636ms/step - loss: 2474.6748 - mae: 43.1385 - val_loss: 2966.3354 - val_mae: 46.2202
Epoch 3/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 410ms/step - loss: 2271.6643 - mae: 42.4237 - val_loss: 2524.5996 - val_mae: 43.5356
Epoch 4/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 383ms/step - loss: 1931.5219 - mae: 38.6989 - val_loss: 2562.7988 - val_mae: 42.5397
Epoch 5/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 468ms/step - loss: 1641.5438 - mae: 35.7183 - val_loss: 2355.9666 - val_mae: 40.8205
Epoch 6/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 496ms/step - loss: 1450.1754 - mae: 33.4343 - val_loss: 2045.2891 - val_mae: 38.2224
Epoch 7/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 391ms/step - loss: 1305.3679 - mae: 31.4918 - val_loss: 1924.1312 - val_mae: 36.6735
Epoch 8/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 615ms/step - loss: 1150.8636 - mae: 29.0424 - val_loss: 1818.3004 - val_mae:

In [ ]:
from google.colab import files
import cv2
import numpy as np
import tensorflow as tf

# 📤 Upload image
uploaded = files.upload()

# Get uploaded file name
img_path = list(uploaded.keys())[0]

# 🔍 Prediction function
def predict_time(img_path):
    # Read image
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))

    # Preprocess
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    img = np.expand_dims(img, axis=0)

    # Predict time
    pred_time = model_time.predict(img)[0][0]

    # Convert to days + hours
    days = int(pred_time // 24)
    hours = int(pred_time % 24)

    print(f"⏳ Estimated Time to Harvest: {days} days {hours} hours")

# 🚀 Run prediction
predict_time(img_path)

Saving IMG_20220820_124012.jpg to IMG_20220820_124012.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
⏳ Estimated Time to Harvest: 3 days 6 hours


In [ ]:
from google.colab import files
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# 📤 Upload image
uploaded = files.upload()
img_path = list(uploaded.keys())[0]

# 🔍 Prediction function
def predict_stage_and_time(img_path):
    # Read image
    img = cv2.imread(img_path)
    img_resized = cv2.resize(img, (224, 224))

    # Preprocess
    img_input = tf.keras.applications.mobilenet_v2.preprocess_input(img_resized)
    img_input = np.expand_dims(img_input, axis=0)

    # -------------------------
    # 🌱 Stage Prediction
    # -------------------------
    stage_pred = model_mobilenet.predict(img_input)
    class_idx = np.argmax(stage_pred)

    class_names = ['Pinhead', 'Growing', 'Mature']
    stage = class_names[class_idx]

    # -------------------------
    # ⏳ Time Prediction
    # -------------------------
    time_pred = model_time.predict(img_input)[0][0]

    days = int(time_pred // 24)
    hours = int(time_pred % 24)

    # -------------------------
    # 📊 Output
    # -------------------------
    print("🌱 Predicted Stage:", stage)

    if stage == "Mature":
        print("✅ Ready to Harvest!")
    else:
        print(f"⏳ Estimated Time: {days} days {hours} hours")

    # Show image
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"Stage: {stage}")
    plt.axis('off')
    plt.show()


# 🚀 Run
predict_stage_and_time(img_path)

In [ ]:
from tensorflow.keras.models import Model

# Remove last classification layer
feature_extractor = Model(inputs=model_mobilenet.input, outputs=model_mobilenet.layers[-2].output)

print("✅ Using second last layer as features")

✅ Using second last layer as features


In [ ]:
X_features = feature_extractor.predict(X)
X_features = X_features.reshape(X_features.shape[0], -1)

print("Feature shape:", X_features.shape)

3/3 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step
Feature shape: (96, 128)


In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_features, y, test_size=0.2, random_state=42
)

svm_model = SVC(kernel='rbf')
svm_model.fit(X_train, y_train)

print("✅ SVM added successfully")

✅ SVM added successfully


In [ ]:
from sklearn.metrics import accuracy_score

y_pred = svm_model.predict(X_test)
print("🎯 Accuracy:", accuracy_score(y_test, y_pred) * 100, "%")

🎯 Accuracy: 65.0 %


In [ ]:
def predict_with_svm(img):
    img = cv2.resize(img, (224,224))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    feature = feature_extractor.predict(img)
    feature = feature.reshape(1, -1)

    pred = svm_model.predict(feature)

    class_names = ['Pinhead', 'Growing', 'Mature']
    return class_names[pred[0]]

In [ ]:
from google.colab import files
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 📤 Upload image
uploaded = files.upload()
img_path = list(uploaded.keys())[0]

# 📸 Read image
img = cv2.imread(img_path)
img_resized = cv2.resize(img, (224, 224))

# 🖼️ Show image
plt.imshow(cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB))
plt.title("Uploaded Image")
plt.axis('off')
plt.show()

# 🔍 Preprocess
img_input = img_resized / 255.0
img_input = np.expand_dims(img_input, axis=0)

# 🎯 Extract features (from your CNN)
feature = feature_extractor.predict(img_input)
feature = feature.reshape(1, -1)

# 🤖 SVM Prediction
pred = svm_model.predict(feature)

class_names = ['Pinhead', 'Growing', 'Mature']
stage = class_names[pred[0]]

# 🖨️ Output
print("🌱 Predicted Stage:", stage)

# ⏳ Optional time logic
if stage == 'Pinhead':
    print("⏳ Estimated Time: 3 days")
elif stage == 'Growing':
    print("⏳ Estimated Time: 1 day")
else:
    print("✅ Ready to Harvest")